# Lecture 3 - Manipulating Data Frames and Calculating Summaries

## Today's Key Takeaways

- **Reshaping** data frames
- Calculating **summary parameters**
- **Sorting** data in `pandas`

### Adding libraries to python

First we need to import the libraries that we'll want to use for today's class. This is the same process you used for the last homework to add a function to python, but libraries can add hundreds of new functions.

When we import these libraries we can assign them an alias, which is easier to remember and type. The ones used below are common for these packages.

In [1]:
# Import required libraries
import numpy as np
import pandas as pd

# Setting pandas options
pd.set_option('display.precision', 2)
pd.set_option('display.max_columns',10)

### From Last Time:

Start by importing the metadata (same as last week).

In [3]:
# Bring in metadata from Excel. You should pick the index column as the column with the most representative sample identifier
meta = pd.read_csv("~/LECTURE_MATERIALS/DataFiles/GSE88741_meta.csv", index_col=1)
meta.head()

,SampleTitle,Stage,CellType
Sample_geo_accession,,,
GSM2344965,FM_1,primary melanocytes,normal melanocytes
GSM2344966,FM_2,primary melanocytes,normal melanocytes
GSM2344967,FM_3,primary melanocytes,normal melanocytes
GSM2344968,SK_MEL_28_1,metastatic,melanoma cell line
GSM2344969,SK_MEL_28_2,metastatic,melanoma cell line


### Subsetting

One of the powerful features of data frames is that rows and columns can be referred to by names or numbers:

In [ ]:
# Extracting data in all rows from the column named "Stage"
meta['Stage']

In [ ]:
# Extracting data in all columns from row "FM_1" (note that you need to add ".loc" in order to refer to the row names)
meta.loc['FM_1']

To select multiple rows or columns, input a list:

In [ ]:
meta.loc[['FM_1','FM_2','FM_3'],'Stage']

You can also subset data by number by using "`.iloc`" (zero-indexing also applies here)

In [ ]:
meta.iloc[0:3,2]

To get Python to output the entire set of row and column names, you can use the `.index` and `.columns` attributes, respectively.

In [ ]:
# listing the row names
meta.index

In [ ]:
# listing the column names
meta.columns

Data frames can also be sliced by rows without explicitly naming the row names or row numbers. Instead, we can slice by testing whether a condition is fulfilled or not:

In [ ]:
# Finding out which rows match the stage: "primary melanocytes"
primary_indices = meta.Stage == 'primary melanocytes'
print(primary_indices)

In [ ]:
# subsetting the rows based on this conditional set
primary_melanocytes_data = meta[primary_indices]
primary_melanocytes_data

You can export back to Excel using the `.to_excel()` method.

In [ ]:
# exporting primary_melanocytes_data back to excel
primary_melanocytes_data.to_excel('primary_melanocytes_data.xlsx',index = False)

## Data Manipulation: adding new columns to data frames

We can add a new column to a data frame using the same syntax as assigning a column slice to a data structure that contains the same number of elements as rows in the data frame.

In [ ]:
# Find out the number of rows and columns with the .shape attribute
meta.shape

`meta` has 12 rows and 3 columns. So say we want to add an extra column to `meta` that just numbers the samples from 1-12.

We can do this by first creating a `numpy` array that contains the numbers 1-12, and then assign this to a new column in the `meta` data frame, that we'll call `sample_number`.

In [ ]:
# Creating a numpy array of that starts from 1 and ends before 13, stepping by 1, using the np.arange() function.
tmp =  np.arange(1,13,1)
print(tmp)

In [ ]:
# Assigning a new column called 'sample_number' in the data frame meta with the values in tmp
meta['sample_number'] = tmp
meta

Let's also create another metadata column, which contains the **cell line** information. Given the way that the samples were labeled in this dataset, we can do this easily by first stripping off the last 2 characters from each of the "Sample Title" row names, and then we can assign the resulting set to a new column, which we can call `cell_line`.

In [ ]:
# create a variable that contains the information in meta.index (the row names)
samples = meta.index
print(samples)

The `sample` variable we just made is a single column of a data frame, which is called a `pandas` `series`. There are functions that are specific to `series`. We can use `series.str` to operate on the strings in the series to remove the last two characters from each element. We can then add that to the `meta` data frame.

In [ ]:
# Remove the last two characters from each sample name
samples = samples.str[:-2]
print(samples)

In [ ]:
# Set a new metadata column to this content, and call it "cell_line"
meta['cell_line'] = samples
meta.head()

In [ ]:
meta

### Reshaping a data frame with `.stack()`, `.pivot`, and `.melt()`

Sometimes, it will be convenient to change how the contents of a data frame is organized in terms of its rows and columns. For some common restructurings, there are built in methods available to reshape the data frame easily.

In [ ]:
# Grouping all of the information by index and column
meta.stack()

In [ ]:
# creating a wide data frame such that the indexes and columns have a specific focus
meta.pivot(index = 'Stage',columns='Sample_geo_accession')

In [ ]:
# pulling everything into a long-form data frame
meta.melt(id_vars = ['Stage','CellLine'])

These reshaping manipulations will be useful for arranging data frames for plotting, as plotting libraries often expect data and plot properties to be represented in separate columns. We'll talk about this more in future classes.

For more information, go [here](https://medium.com/@prathik.codes/reshaping-pandas-dataframes-melt-vs-stack-vs-pivot-vs-explode-7a47caf833c6) and [here](https://pandas.pydata.org/docs/user_guide/reshaping.html).

### Adjusting data frames for intuitive referencing of data **subsets**
To showcase some additional data frame functionalities, let's also import a dataset that contains numbers.

We also want to make a data frame that contains the expression count data, imported from the `GSE88741-expression.csv` file. We'll store the expression counts dataset in the data frame variable called `df`.

In [5]:
# Import comma-seperated data from a text file
df = pd.read_csv('~/LECTURE_MATERIALS/DataFiles/GSE88741-expression-B.csv', index_col=0)
df.head()

,GSM2344965,GSM2344966,GSM2344967,GSM2344968,GSM2344969,...,GSM2344972,GSM2344973,GSM2344974,GSM2344975,GSM2344976
gene_symbol,,,,,,,,,,,
A1BG,400,320,490,331,363,...,248,301,755,391,310
A1CF,1,1,3,0,0,...,2,3,1,0,5
A2M,23278,47606,20484,2652,2707,...,7,3,26726,10394,12096
A2ML1,6,8,10,1,7,...,0,1,8,4,7
A2MP1,21,7,34,0,6,...,0,0,13,3,3


We want to rearrange the data frame `df` to follow the `pandas` data frame convention: that different replicates are represented in the rows and different variables (genes) are represented in the columns. To do this, we'll transpose the data frame with the `.transpose()` method.

In [ ]:
# Transposing the data frame
df = df.transpose()
df.head()

One of the powerful features of data frames is that you can refer to rows by the identifier in the index column. To take advantage of this utility, we want to make the contents of the index column more informative/relevant to what these specific samples represent.

The "Sample Titles" column in the `meta` data frame contains sample identifiers that are informative than the 'GSM...' identifiers that currently represent the index in `df`. Our goal will then be to change the index of `df` to match the contents of the Sample Titles column from `meta`.


In [ ]:
# Extract the Sample Titles and use them to replace the ugly GSM names
df.index = meta.index
df.head()

This allows us to look up gene counts for a specific gene name and sample title.

In [ ]:
# Finding out the expression of the gene A2M is in the sample FM_1
print(df.loc['FM_1','A2M'])

### Data Manipulation: remove columns with `NA` or `NaN` values

Sometimes, datasets will contain entries with `NA` or `NaN`, which represet the absence of specific values. We want to get rid of rows with these values. We can do this easily with`.dropna()` (reference [here](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.dropna.html)):

In [ ]:
# removing NAs and NaNs
df.dropna()

### Data Manipulation: remove columns with low counts by conditional slicing

Often times when working with high-throughput experimental data, there will be specific columns (corresponding to variables) that we won't want to consider, because those columns contain data that don't pass a particular inclusion criteria. This case study (removing columns with low counts), will show how to remove the columns that don't pass the inclusion criteria from the data frame without having to know beforehand which column numbers they are.

In [ ]:
# Make a Data Frame of boolean values showing where there are read counts less than 1
df_low = df < 1
df_low

In [ ]:
# Add together all of the columns to get total number of samples with low counts
lowsum = df_low.sum()
lowsum.head()

In [ ]:
# Showing which samples have low expression of the gene A1CF
df_low.A1CF

In [ ]:
# Subsetting the expression matrix on the samples deemed to be low in A1CF expression.
df.loc[df_low.A1CF,'A1CF']

To help us get a sense of the distribution of the `lowsum` variable, we can use `pandas` calculation of summary parameters using the `.describe()` method.

In [ ]:
# Looking at the quantiles
lowsum.describe()

You can see that many genes either have reads for all 12 samples or none of them. We want to remove all of those genes from our data frame. Let's do that by selecting on a list of genes that have reads for all 12 samples.

In [ ]:
# Make a mask of all the columns we want to keep
# Let's be picky and only accept genes where all 12 samples have counts
keep = lowsum < 1
keep.head()

In [ ]:
# Calculating how many are have reads for all 12 samples
print(np.sum(keep))

In [ ]:
# Using df.loc we can select just the genes we want to keep
df_low_removed = df.loc[:,keep]

In [ ]:
# Check the shape before and after
print(df.shape)
print(df_low_removed.shape)

In [ ]:
df_low_removed.head()

## <font color=brown>Hands on practice</font>
How many genes have less than 12 samples with reads, but more than zero?

### Data Manipulation: log2-transform expression counts

Another useful manipulation of data is to apply a mathematical operation uniformly across all of the values in a dataset. In this case study, we're going to apply the log2 transformation.

In [ ]:
# Applying the log2 transformation
df_low_removed_log2 = np.log2(df_low_removed)

df_low_removed_log2.head()

### Data Manipulation: Joining data frames

Let's join the `meta` data frame with the `df` data frame. To do this, we will use the `pandas` function `pd.concat()`. Note that this works similarly to `.hstack()` and `.vstack()` from numpy, but you must specify which axis to concatenate (0 = stitch by rows, equivalent to `vstack`, 1 = stitch by columns, equivalent to `hstack`).

In [ ]:
# Merge the two data frames with concat
melanoma_log2 = pd.concat([meta,df_low_removed_log2],axis = 1)

In [ ]:
# Check the dimensions and show the first few lines of the new data frame
melanoma_log2.shape
melanoma_log2.head()

In [ ]:
# Calling on a specific column in the new data frame
melanoma_log2['CellType']

Let's export this data frame to excel, so that we can directly import this data frame in future classes.

In [ ]:
# export to excel, including the index column
melanoma_log2.to_excel('melanoma_zerosRemoved_log2transformed.xlsx',index = True)

## Calculating Summary Parameters

`pandas` also enables calculation of summary parameters using the `.describe()` method (reference [here](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.describe.html)). Note that this will only work effectively for small data frames, so in this example, we are only looking at the first 10 genes of `df_low_removed_log2` (we're using `df_low_removed_log2` instead of `melanoma_log2` because it doesn't have the metadata columns):

In [ ]:
df_low_removed_log2.iloc[:,:10].describe()

You can also customize what your summary statistics will consist of with the `.aggregate()` method (reference [here](https://www.geeksforgeeks.org/python-pandas-dataframe-aggregate/) and [here](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.aggregate.html)).

In [ ]:
# Calculating the minimum, median, max, and sum across rows of each column.
# Note that the median function is found in numpy.
df_low_removed_log2.iloc[:,:10].aggregate(['min','median','max','sum'])

The set of aggregate metrics that can be calculated are:

- `mean` - the average
- `median` - the median
- `prod` - the product
- `sum` - the sum
- `std` - the standard deviation
- `var` - the variance

### Calculating overall metrics and sorting

For each of the metrics that you can use with `.aggregate()` (e.g. `df.mean()`, `df.min()`, etc.), you can calculate them individually.  

This is particularly useful when you want to find genes that are doing interesting things in your data.

In [ ]:
# Calculate the overall variance with df.var()
overall_variance = melanoma_log2.var(numeric_only=True)
print(overall_variance.head())
print(overall_variance.shape)

**Case study**: we want to find genes that are highly variable across conditions, but we don't know which they are. Therefore, we're going to sort the overall variance values that we just calculated in descending order (i.e. from largest to smallest) using the `.sort_values()` method. Then, we'll be able to pick out which genes are the most variable

In [ ]:
# Sort this Series and select the 20 genes with the largest variance
overall_variance.sort_values(inplace = True, ascending= False)
overall_variance.head()

Notice that the `overall_variance` variable has changed, even though we haven't reassigned it (with the = sign). This is because we set the `.sort_values()` argument `inplace = True`.

In [ ]:
# Now print the top 10 most variable genes.
topvar = overall_variance[:10]
print(topvar)
type(topvar)

We can pull out the gene information by accessing the `index` attribute of this `pandas series`:

In [ ]:
# We want to use the gene names, not their variance, to filter the columns of melanoma_log2
topvar.index

We can then use these indexes to slice the `melanoma_df` data frame and look at what the actual expression count values were for these genes.

In [ ]:
# Now we can use that Series with melanoma_log.loc to make a subtable
topvartable = melanoma_log2.loc[:,topvar.index]
topvartable

## Computation based on sample groups

Our expression dataset has three replicates for each cell line. `pandas` enables calculation of multiple paramaters with these replicates consolidated, if your input data frame contains a variable that contains the grouping information.

The replicates can be grouped together with the syntax:
```python
df.groupby(['column_name'])
```

In [ ]:
# reminder of what melanoma_log2 looks like
melanoma_log2.head()

In [ ]:
# using .groupby() to designate groupings by cell line
mel_by_cel = melanoma_log2.groupby('cell_line')
mel_by_cel

We have made a "`DataFrameGroupBy`" object.

This is an iterator, an object that iterates over a function, offering it one block of data at a time. To generate the mean of each gene for each cell line, we use the following:

In [ ]:
# calculates group-by mean
mel_by_cel_mean = mel_by_cel.mean(numeric_only=True)
mel_by_cel_mean

In [ ]:
# calculates group-by variance
mel_by_cel_var = mel_by_cel.var(numeric_only=True)
mel_by_cel_var

 We can similarly calculate the `median`, `min`, `max`, `var`, and a host of other metrics.

### <font color=brown>Hands on practice</font>
1. Use `groupby` to calculate the mean expression counts across each stage.
2. Use `describe` to calculate the summary parameters of `mean_bystage` for the metastatic and primary melanocytes stages separately.
3. Find the top 10 variably expressed gene between the stages from the `mean_bystage` variable.
4. Display the `mean_bystage` table sliced to show only these top 10 genes.

In [ ]:
mean_bystage =


For more information about grouping, summarization, and aggregation, [go here](https://jakevdp.github.io/PythonDataScienceHandbook/03.08-aggregation-and-grouping.html).